# Exercice : Cycle de Développement de Prompt

Ce notebook vous guide à travers le processus de création, d'évaluation et de raffinement de prompts pour transformer un texte technique complexe en contenu de formation simple pour les employés.

### Concepts clés :
*   **Contrôle du ton, de la structure et du format.**
*   **Mitigation des hallucinations.**
*   **Reformulation vs Citation.**

In [2]:
# Configuration de l'API Gemini
import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Mise à jour vers un modèle confirmé disponible dans votre environnement
    model_name = 'models/gemini-2.0-flash'
    model = genai.GenerativeModel(model_name)

    print(f"Configuration réussie ! Modèle '{model_name}' prêt pour l'exercice.")
except Exception as e:
    print(f"Erreur de configuration : {e}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Configuration réussie ! Modèle 'models/gemini-2.0-flash' prêt pour l'exercice.


## Étape 1 : Création d'un Prompt de Contrôle

L'objectif ici est de transformer un texte dense et formel en un court message amical. Un bon prompt doit spécifier :
1. **Le rôle** (Équipe Communication).
2. **La tâche** (Réécrire un texte).
3. **Les contraintes** (Moins de 75 mots, puces, pas de citations directes).

In [4]:
import time

policy_text = """Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications."""

prompt_step1 = f"""
Tu travailles dans l'équipe Learning & Communications.
Transforme le texte de politique suivant en un court extrait pour une newsletter interne.

Contraintes :
- Ton : Amical et clair.
- Format : Utilise des listes à puces.
- Méthode : Reformule tout, ne cite pas directement le texte.
- Longueur : Maximum 75 mots.

Texte source :
{policy_text}
"""

try:
    # Tentative de génération
    response = model.generate_content(prompt_step1)
    print("Résultat du Prompt Step 1 :\n")
    print(response.text)
except Exception as e:
    if "429" in str(e):
        print("Erreur 429 : Quota encore dépassé. Attendez encore quelques secondes avant de relancer.")
    else:
        print(f"Une erreur est survenue : {e}")

Erreur 429 : Quota encore dépassé. Attendez encore quelques secondes avant de relancer.


## Étape 2 & 3 : Évaluation et Mitigation des Hallucinations

Si le modèle a inventé des détails (comme mentionner un logiciel spécifique non présent dans le texte), nous devons restreindre sa créativité.

In [18]:
prompt_mitigation = f"""
Tu es un expert en sécurité. Reformule le texte ci-dessous pour les employés.

STRICTES INSTRUCTIONS :
- Utilise uniquement les informations fournies dans le texte source.
- Ne suggère pas de nouvelles technologies ou recommandations.
- Ton amical, liste à puces, moins de 75 mots.

Texte source :
{policy_text}
"""

try:
    response_refined = model.generate_content(prompt_mitigation)
    print("Résultat raffiné (sans hallucinations) :\n")
    print(response_refined.text)
except Exception as e:
    if "429" in str(e):
        print("Erreur 429 : Limite de requêtes atteinte. Veuillez patienter une minute avant de relancer cette cellule.")
    else:
        print(f"Une erreur est survenue : {e}")

Erreur 429 : Limite de requêtes atteinte. Veuillez patienter une minute avant de relancer cette cellule.


## Étape 4 : Adaptation à une audience spécifique (Stagiaires)

Ici, on simplifie encore plus en utilisant un langage de soutien.

In [19]:
prompt_interns = f"""
Explique cette règle de sécurité à un stagiaire junior.

Instructions :
- Utilise un ton encourageant et informatif.
- Maximum 4 puces.
- Évite tout jargon corporatif ou légal.
- Utilise des phrases courtes.

Texte source :
{policy_text}
"""

try:
    response_interns = model.generate_content(prompt_interns)
    print("Version pour stagiaires :\n")
    print(response_interns.text)
except Exception as e:
    if "429" in str(e):
        print("Erreur 429 : Quota atteint. Attendez 30-60 secondes avant de relancer.")
    else:
        print(f"Une erreur est survenue : {e}")

Erreur 429 : Quota atteint. Attendez 30-60 secondes avant de relancer.


## Étape 5 : Extraction de Citations et Réflexion

Parfois, il vaut mieux citer le texte original pour garder l'autorité de la règle.

### Exercice :
1. Extraire la citation la plus importante.
2. Répondre aux questions sur l'usage des citations.

In [20]:
prompt_quote = f"""
Extrais une seule citation directe du texte suivant qui capture l'essence même de la politique de sécurité.

Texte source :
{policy_text}
"""

try:
    response_quote = model.generate_content(prompt_quote)
    print("Citation extraite :\n")
    print(response_quote.text)
except Exception as e:
    if "429" in str(e):
        print("Erreur 429 : Quota atteint. Attendez 30-60 secondes.")
    else:
        print(f"Une erreur est survenue : {e}")

print("\n--- Questions de réflexion ---")
print("1. Quand utiliser une citation ? Dans des documents officiels, contrats ou avis de sécurité critiques pour éviter toute ambiguïté.")
print("2. Risque des citations ? Elles peuvent rester trop techniques (jargon) et être moins bien comprises par les non-experts.")

Erreur 429 : Quota atteint. Attendez 30-60 secondes.

--- Questions de réflexion ---
1. Quand utiliser une citation ? Dans des documents officiels, contrats ou avis de sécurité critiques pour éviter toute ambiguïté.
2. Risque des citations ? Elles peuvent rester trop techniques (jargon) et être moins bien comprises par les non-experts.
